# Causal Inference in Practice
## Week 10 — Difference-in-Differences & Panel Methods · Practice Notebook

> **Block III — Quasi-experimental designs**
>
> Let a comparison group subtract out everything that would have happened anyway.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · A panel with a known effect

Difference-in-differences would be untestable in the wild — we never see the treated group's untreated counterfactual. So we **simulate** a panel where we control everything: unit fixed effects (some units are just higher), a common time trend (everything drifts up together), and a treatment effect we **switch on** for treated units in the post period. Because we built it, we know the true effect — our answer key for the whole notebook.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

N, TRUE_ATT = 600, 3.0            # units, and the known effect
treated = (np.arange(N) < N // 2).astype(int)   # half are treated
unit_fe = RNG.normal(0, 2, N)     # fixed, time-invariant unit levels
time_fe = {0: 0.0, 1: 1.5}        # common trend: both groups drift up

rows = []
for i in range(N):
    for t in (0, 1):              # t=0 pre, t=1 post
        post = t
        y = (5 + unit_fe[i] + time_fe[t]
             + TRUE_ATT * treated[i] * post      # effect only when treated AND post
             + RNG.normal(0, 1))
        rows.append((i, t, treated[i], post, y))
panel = pd.DataFrame(rows, columns=['unit','time','treated','post','y'])
print(panel.head())
print(f'\nTrue effect built into the treated post cells: {TRUE_ATT}')

Note the structure: treated and control start at **different levels** (unit fixed effects differ) and both move up by the same common trend. Only the treated-and-post cells get the extra `TRUE_ATT`. That is exactly the world DiD is designed for.

## 2 · The 2×2 estimator recovers the effect

Take the four group×period **cell means** and difference twice. The first difference (after − before, within a group) removes that group's fixed level. The second difference (treated − control) removes the common time trend. What's left is the effect.

In [ ]:
cell = panel.groupby(['treated','post'])['y'].mean().unstack()
print('Cell means (rows=treated, cols=post):')
print(cell, '\n')

d_treated = cell.loc[1, 1] - cell.loc[1, 0]   # treated change over time
d_control = cell.loc[0, 1] - cell.loc[0, 0]   # control change over time
did = d_treated - d_control
print(f'treated change = {d_treated:.3f}')
print(f'control change = {d_control:.3f}   (= the common trend)')
print(f'2x2 DiD        = {did:.3f}   (true {TRUE_ATT})')
assert abs(did - TRUE_ATT) < 0.3, 'the 2x2 DiD should recover the effect'

The control group's change (~1.5) is our estimate of *what would have happened to the treated group anyway*. Subtracting it from the treated change leaves the effect — about 3.0. The different starting levels never mattered; they cancelled in the first difference.

## 3 · The same number from a TWFE regression

DiD is equivalently the **interaction coefficient** `treated:post` in a regression. With unit and time fixed effects (here, in the 2-period case, the `treated` and `post` main effects play that role), the `treated:post` term is the difference-in-differences. We also attach **cluster-robust standard errors at the unit level** — the right inference for panel data.

In [ ]:
fit = smf.ols('y ~ treated + post + treated:post', data=panel).fit(
    cov_type='cluster', cov_kwds={'groups': panel['unit']})
b  = fit.params['treated:post']
se = fit.bse['treated:post']
print(f'TWFE interaction = {b:.3f}   (true {TRUE_ATT})')
print(f'cluster-robust SE = {se:.3f}   (clustered by unit)')
assert abs(b - did) < 1e-8, 'the interaction coef IS the 2x2 DiD'
assert abs(b - TRUE_ATT) < 0.3, 'TWFE should recover the effect'

The interaction coefficient is **numerically identical** to the hand-computed 2×2 DiD — the regression is just a convenient way to get the same number plus a standard error. Clustering by unit allows a unit's periods to be correlated; ignoring that would make the SE too small.

### 🔧 Exercise 3.1 — break parallel trends, break DiD

DiD is only as good as parallel trends. Re-simulate the panel but give the **treated** group an extra upward drift in the post period that has *nothing to do with treatment* (a differential trend). The true treatment effect is still `3.0`, but DiD will now **overstate** it, because it credits the differential trend to the treatment.

Fill in the `# TODO`s: add a `bad_trend` only to treated-and-post cells, then recompute the 2×2 DiD.

In [ ]:
# TODO: rebuild the panel with a differential (non-parallel) trend.
BAD = 2.0   # extra treated-only post drift, NOT a treatment effect
rows_bad = []
for i in range(N):
    for t in (0, 1):
        bad_trend = 0.0   # TODO: = BAD only when treated[i]==1 and t==1, else 0
        y = (5 + unit_fe[i] + time_fe[t]
             + TRUE_ATT * treated[i] * t
             + bad_trend + RNG.normal(0, 1))
        rows_bad.append((i, t, treated[i], t, y))
# bad = pd.DataFrame(rows_bad, columns=['unit','time','treated','post','y'])
# ... then compute the 2x2 DiD on `bad` and compare to TRUE_ATT
print('fill in bad_trend, then compute the DiD (see the solution cell)')

### ✅ Solution 3.1

In [ ]:
BAD = 2.0
rows_bad = []
for i in range(N):
    for t in (0, 1):
        bad_trend = BAD if (treated[i] == 1 and t == 1) else 0.0
        y = (5 + unit_fe[i] + time_fe[t]
             + TRUE_ATT * treated[i] * t
             + bad_trend + RNG.normal(0, 1))
        rows_bad.append((i, t, treated[i], t, y))
bad = pd.DataFrame(rows_bad, columns=['unit','time','treated','post','y'])

cb = bad.groupby(['treated','post'])['y'].mean().unstack()
did_bad = (cb.loc[1,1]-cb.loc[1,0]) - (cb.loc[0,1]-cb.loc[0,0])
print(f'DiD under broken parallel trends = {did_bad:.3f}  (true {TRUE_ATT})')
print(f'overstatement = {did_bad - TRUE_ATT:+.3f}  (≈ the differential trend, {BAD})')
assert did_bad - TRUE_ATT > 1.0, 'broken parallel trends should bias DiD upward'

The estimate is now ~5 instead of 3: DiD blamed the treatment for a trend the treated group would have had anyway. **No amount of data fixes this** — it is an identification failure, not noise. That is why we probe parallel trends with an event study next.

## 4 · Event-study: leads diagnose, lags describe

With several pre- and post-periods we can estimate **one coefficient per period relative to adoption**, omitting the period just before treatment (`t = −1`) as the reference. The pre-period 'leads' should sit near zero (parallel trends); the post-period 'lags' trace the **dynamic** effect — here it ramps up and plateaus.

In [ ]:
# A richer panel: 8 periods, treated units adopt at g=4, dynamic effect.
Tn, g, PEAK = 8, 4, 2.5
u_fe = RNG.normal(0, 2, N)
t_fe = np.linspace(0, 3, Tn)          # common trend, identical for both groups
rows_es = []
for i in range(N):
    for t in range(Tn):
        if treated[i] == 1 and t >= g:
            eff = PEAK * min(1.0, 0.5 * (t - g + 1))   # ramps 1.25→2.5→plateau
        else:
            eff = 0.0
        y = 5 + u_fe[i] + t_fe[t] + eff + RNG.normal(0, 1)
        rows_es.append((i, t, treated[i], y))
es = pd.DataFrame(rows_es, columns=['unit','time','treated','y'])
es['evt'] = es['time'] - g            # event time (0 = adoption period)
print(es.head())

In [ ]:
def event_study(d):
    """OLS event-study: treated×(event-time==k) dummies + unit & time FE.
    Omits k=-1 as the reference. Returns {event_time: (coef, se)}."""
    levels = [k for k in range(int(d['evt'].min()), int(d['evt'].max()) + 1)
              if k != -1]
    dummies = {f'ev{k:+d}': ((d['treated'] == 1) & (d['evt'] == k)).astype(float).values
               for k in levels}
    unit_d = pd.get_dummies(d['unit'], prefix='u', drop_first=True).astype(float)
    time_d = pd.get_dummies(d['time'], prefix='t', drop_first=True).astype(float)
    X = sm.add_constant(pd.concat(
        [pd.DataFrame(dummies, index=d.index), unit_d, time_d], axis=1))
    m = sm.OLS(d['y'].values, X.values).fit(
        cov_type='cluster', cov_kwds={'groups': d['unit'].values})
    coef = dict(zip(X.columns, m.params)); se = dict(zip(X.columns, m.bse))
    return {k: (coef[f'ev{k:+d}'], se[f'ev{k:+d}']) for k in levels}

pts = event_study(es)
for k in sorted(pts):
    c, s = pts[k]
    print(f'event time {k:+d}:  coef {c:+.3f}  (SE {s:.3f})')

In [ ]:
# Plot the event study: leads (k<0) near 0, lags (k>=0) trace the effect.
ks  = sorted(pts)
bs  = np.array([pts[k][0] for k in ks])
ses = np.array([pts[k][1] for k in ks])

fig, ax = plt.subplots()
ax.errorbar(ks, bs, yerr=1.96 * ses, marker='o', capsize=3, color='#4C72B0')
ax.axhline(0, color='grey', lw=1)
ax.axvline(-0.5, color='crimson', ls='--', lw=1, label='adoption')
ax.set_xlabel('event time (periods relative to adoption)')
ax.set_ylabel('coefficient (vs t = -1)')
ax.set_title('Event study: flat leads, rising lags'); ax.legend()
None  # figure created; no blocking show()

leads = np.array([pts[k][0] for k in ks if k < 0])
print(f'max |lead| = {np.abs(leads).max():.3f}  (≈0 ⇒ parallel pre-trends)')
assert np.abs(leads).max() < 0.4, 'pre-period leads should be ~0'

The leads hug zero — the visual parallel-trends check passes — and the lags climb from ~1.25 to ~2.5 and flatten, exactly the dynamic effect we built in. This single plot is both the diagnostic and the story of how the effect unfolds.

### 🔧 Exercise 4.1 — make the pre-trends fail

Add a treated-group drift that starts *before* adoption (a differential pre-trend). Re-run `event_study` and confirm the **leads are no longer flat** — the plot now warns you that the design is shaky.

Fill in the `# TODO`: add `0.6 * es['evt']` worth of drift to treated units for *all* periods (so it bends the pre-period too).

In [ ]:
# TODO: build a panel where treated units drift with event time everywhere.
es_bad = es.copy()
drift = ...   # TODO: 0.6 * es_bad['evt'] when treated==1, else 0
# es_bad['y'] = es_bad['y'] + drift
# pts_bad = event_study(es_bad)
# leads_bad = [pts_bad[k][0] for k in pts_bad if k < 0]
# print('pre-period leads:', [round(v,2) for v in leads_bad])
print('fill in the drift and re-run the event study')

### ✅ Solution 4.1

In [ ]:
es_bad = es.copy()
drift = np.where(es_bad['treated'] == 1, 0.6 * es_bad['evt'], 0.0)
es_bad['y'] = es_bad['y'] + drift
pts_bad = event_study(es_bad)
leads_bad = np.array([pts_bad[k][0] for k in sorted(pts_bad) if k < 0])
print('pre-period leads (should NOT be flat):',
      [round(v, 2) for v in leads_bad])
print(f'max |lead| now = {np.abs(leads_bad).max():.3f}  (was ~0)')
assert np.abs(leads_bad).max() > 0.5, 'a differential pre-trend should bend the leads'

Now the leads slope away from zero: the event-study plot is *screaming* that treated and control were already diverging before treatment. This is exactly the red flag the diagnostic exists to raise — you would not trust a DiD effect read off this design.

## 5 · Staggered adoption: TWFE breaks, group-time fixes it

Now the hard case. Three cohorts adopt at **different times** with **heterogeneous, growing** effects: an early cohort with a big effect, a late cohort with a small one, and a never-treated group. The naive 'currently treated' TWFE coefficient uses already-treated early adopters as 'controls' for the late adopters (forbidden comparisons) and gets the wrong answer. A clean **group-time** estimate (Callaway–Sant'Anna style) that compares each cohort only to the never-treated recovers the truth.

In [ ]:
# Three cohorts: early (g=3, big effect), late (g=8, small), never-treated.
Tn = 12
Np = 150
g_of, ufe2 = {}, {}
uid = 0
for cohort_g in (3, 8, np.inf):           # inf = never treated
    for _ in range(Np):
        g_of[uid] = cohort_g
        ufe2[uid] = RNG.normal(0, 2)
        uid += 1
Nstag = uid
t_fe2 = np.linspace(0, 4, Tn)             # common trend

def true_effect(i, t):
    g = g_of[i]
    if t < g:
        return 0.0
    base = {3: 4.0, 8: 1.0}[g]            # heterogeneous size by cohort
    return base * (1.0 + 0.15 * (t - g))  # grows with time since adoption

rows_s = []
for i in range(Nstag):
    for t in range(Tn):
        y = 5 + ufe2[i] + t_fe2[t] + true_effect(i, t) + RNG.normal(0, 1)
        rows_s.append((i, t, g_of[i], y))
stag = pd.DataFrame(rows_s, columns=['unit','time','g','y'])
stag['treated_now'] = (stag['time'] >= stag['g']).astype(float)
print(stag.head())

In [ ]:
# The TRUE answer: average effect over all actually-treated unit-periods.
treated_cells = stag[stag['time'] >= stag['g']]
TRUE_AVG_ATT = np.mean([true_effect(r.unit, r.time)
                        for r in treated_cells.itertuples()])
print(f'TRUE average ATT over treated cells = {TRUE_AVG_ATT:.3f}')

# --- Naive TWFE: y ~ treated_now + unit FE + time FE (cluster by unit) ---
unit_d = pd.get_dummies(stag['unit'], prefix='u', drop_first=True).astype(float)
time_d = pd.get_dummies(stag['time'], prefix='t', drop_first=True).astype(float)
X = sm.add_constant(pd.concat(
    [stag[['treated_now']].reset_index(drop=True), unit_d, time_d], axis=1))
mtwfe = sm.OLS(stag['y'].values, X.values).fit(
    cov_type='cluster', cov_kwds={'groups': stag['unit'].values})
twfe = mtwfe.params[list(X.columns).index('treated_now')]
print(f'Naive TWFE coefficient = {twfe:.3f}')
print(f'TWFE bias = {twfe - TRUE_AVG_ATT:+.3f}   (it misses the truth!)')
assert abs(twfe - TRUE_AVG_ATT) > 0.5, 'naive TWFE should be visibly biased here'

The naive TWFE coefficient is well **below** the true average ATT. The culprit: when the late cohort switches on, the early cohort is already treated and its effect is still *growing* — using it as a 'control' subtracts that growth, dragging the estimate down (the negative-weights / forbidden-comparison problem). Now the clean fix.

In [ ]:
# --- Clean group-time ATT(g,t): each cohort vs the NEVER-TREATED only. ---
# For cohort g and post period t>=g, baseline is the period just before (g-1):
#   ATT(g,t) = [ȳ_g(t) - ȳ_g(g-1)] - [ȳ_never(t) - ȳ_never(g-1)]
ybar_never = stag[~np.isfinite(stag['g'])].groupby('time')['y'].mean()

att_gt, weights = [], []
for g in (3, 8):
    cohort = stag[stag['g'] == g]
    ybar_g = cohort.groupby('time')['y'].mean()
    base = g - 1                       # last clean pre-period for this cohort
    for t in range(g, Tn):
        gt = (ybar_g[t] - ybar_g[base]) - (ybar_never[t] - ybar_never[base])
        att_gt.append(gt)
        weights.append((cohort['time'] == t).sum())   # # treated cells

cs_att = np.average(att_gt, weights=weights)
print(f'Clean group-time ATT = {cs_att:.3f}   (true {TRUE_AVG_ATT:.3f})')
print(f'group-time bias = {cs_att - TRUE_AVG_ATT:+.3f}   (≈ 0 — recovered!)')
assert abs(cs_att - TRUE_AVG_ATT) < 0.3, 'group-time should recover the true ATT'

Same data, same true effect — but the clean group-time estimator lands on the truth while naive TWFE does not. The only difference is **which comparisons are allowed**: the group-time method never uses an already-treated unit as a control. That is the entire insight behind Callaway–Sant'Anna and the modern DiD literature.

### 🔧 Exercise 5.1 — never-treated vs not-yet-treated controls

Our clean estimate used the **never-treated** as the comparison group. A valid alternative is the **not-yet-treated**: at period `t`, any unit with `g > t` is still a clean control. For the **early** cohort (`g = 3`) at period `t = 5`, the late cohort (`g = 8`) is not-yet-treated and usable. Compute `ATT(3, 5)` using the late cohort (plus never-treated) as the control, and check it recovers the true effect for that cell.

Fill in the `# TODO`s.

In [ ]:
g_e, t_star = 3, 5
base_e = g_e - 1
early = stag[stag['g'] == g_e]
# not-yet-treated at t_star: units whose adoption time g > t_star
ctrl = stag[stag['g'] > t_star]      # late cohort (g=8) + never-treated

# TODO: compute the clean 2x2 DiD for the early cohort at t_star using `ctrl`.
ybar_e    = early.groupby('time')['y'].mean()
ybar_ctrl = ctrl.groupby('time')['y'].mean()
att_35 = ...   # TODO: (ybar_e[t_star]-ybar_e[base_e]) - (ctrl change over same periods)
# truth = true_effect for an early-cohort unit at t_star
# print(att_35, true_effect_for_early_at_t_star)

### ✅ Solution 5.1

In [ ]:
ybar_e    = early.groupby('time')['y'].mean()
ybar_ctrl = ctrl.groupby('time')['y'].mean()
att_35 = (ybar_e[t_star] - ybar_e[base_e]) \
         - (ybar_ctrl[t_star] - ybar_ctrl[base_e])

truth_35 = 4.0 * (1.0 + 0.15 * (t_star - g_e))   # true_effect for g=3 at t=5
print(f'ATT(3, 5) via not-yet-treated controls = {att_35:.3f}')
print(f'true effect for early cohort at t=5     = {truth_35:.3f}')
assert abs(att_35 - truth_35) < 0.4, 'not-yet-treated control should also be clean'

Both control choices — never-treated and not-yet-treated — give a clean comparison, because in neither case is the control group's own treatment effect contaminating the contrast. The forbidden comparison only arises when you use an **already-treated** group as a control, which is exactly what naive TWFE does under the hood.

## Wrap-up & self-check

- **2×2 DiD** = (treated change) − (control change); the two differences remove the fixed level gap and the common trend, recovering the effect.
- The TWFE **interaction coefficient** equals the 2×2 DiD, and we cluster standard errors at the unit because panel rows are serially correlated.
- **Parallel trends** is the whole assumption. You broke it and watched DiD overstate the effect, then saw an **event study** flag a differential pre-trend through non-flat leads.
- Under **staggered adoption** with heterogeneous, dynamic effects, naive TWFE is biased (forbidden comparisons / negative weights); a clean **group-time** estimate using never-treated (or not-yet-treated) controls recovers the true ATT.

**You're ready for Week 11** if you can compute a 2×2 by hand, read an event-study plot, and explain why TWFE breaks under staggered adoption. Next week: **synthetic control** — when you have one treated unit and many candidate controls, build a bespoke comparison unit from a weighted blend of the others.